[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JulesMalin/isba2411-nlp/blob/main/Week%207/L13_Game_Shipping_Decision.ipynb)

# 🏁 The Shipping Decision
### ISBA 2411 · Week 7 · in-class team competition

---

You have built a support copilot that reads Cobalt's queue and sorts each ticket into one of eight
buckets. It is right about **64%** of the time overall, and it tells you **how confident** it is on
every single ticket.

Now you have to ship it. That means answering one question:

> ### How sure does the copilot have to be before we let it route a ticket without a human?

Set the bar **low** and you automate almost everything, including the ones it gets wrong.
Set the bar **high** and you barely automate anything, but you are almost never wrong.

There is no setting that is good at both. **Your job is to find the best setting for your company.**

---

### How the competition works

1. Enter your **team number**. You will be given a company, and what mistakes cost that company.
2. Try different values of the confidence bar and watch your **weekly cost** move.
3. When you are happy, run the submit cell and read your line out to the class.
4. **Scoring:** every company has a different bill, so raw dollars are not comparable.
   You are scored on **how close you get to the best possible setting for your own company**.
   The team that leaves the least money on the table wins.

Each team has a **different company**. Do not copy another team's number: it will be wrong for you.

## Step 1 · Set up (about 10 seconds, no GPU needed)

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt

routed = pd.read_csv("https://raw.githubusercontent.com/JulesMalin/isba2411-nlp/main/data/cobalt_routed.csv")
TICKETS_PER_WEEK = 5000

print(f"{len(routed)} tickets, already routed by the copilot")
routed[["ticket_text","category","predicted","confidence"]].head(4)

Each row is one ticket: what the customer wrote, what it **really** was, what the copilot **guessed**,
and how **confident** the copilot was. Confidence runs from 0 to 1.

## Step 2 · Meet your company

In [ ]:
TEAM = 1          # <-- PUT YOUR TEAM NUMBER HERE (1 to 6), then run this cell

COMPANIES = {
 1: ("Pixelbloom",          "A consumer photo app with 2 million users. Support is a cost centre and "
                            "nobody escalates a mis-sorted ticket, they just re-send it.",
     7,  6,  0.92),
 2: ("Kettle",              "A four-person startup. The two founders answer tickets themselves, at "
                            "midnight, instead of building the product.",
     22, 14, 0.90),
 3: ("Vantage Retail",      "A mid-market SaaS company with an ordinary support desk. Nothing unusual "
                            "in either direction.",
     20, 8,  0.95),
 4: ("Northwind Logistics", "Enterprise customers with contractual response-time SLAs. A ticket that "
                            "sits in the wrong queue breaches the contract.",
     36, 8,  0.34),
 5: ("Meridian Bank",       "A regulated bank. Every mis-sorted ticket is a finding in the next audit, "
                            "and findings cost real money.",
     65, 10, 0.34),
 6: ("Halcyon Health",      "Healthcare. Tickets contain patient data, so a ticket sent to the wrong "
                            "team is a reportable privacy incident.",
     110, 11, 0.32),
}

NAME, STORY, COST_WRONG, COST_HUMAN, START_BAR = COMPANIES[TEAM]
print(f"TEAM {TEAM}: {NAME}\n")
print(STORY, "\n")
print(f"  Cost when the copilot routes a ticket to the WRONG team   ${COST_WRONG}")
print(f"  Cost when a human has to read and sort a ticket instead   ${COST_HUMAN}")
print(f"  Tickets per week                                          {TICKETS_PER_WEEK:,}")
print(f"\n  Your copilot is currently set to a confidence bar of {START_BAR:.2f}.")
print(f"  That setting was picked by nobody in particular. Your job is to do better.")

### Where those two numbers come from

**Cost of a wrong route.** The ticket lands with the wrong team, sits there, gets bounced, and the
customer waits. For a consumer app that is mild annoyance. For a bank it is an audit finding. For a
hospital it is a privacy incident. That is why the number is different for every company here.

**Cost of a human sorting it.** An agent reads the ticket and decides where it goes. A few minutes of
salary. Higher at a startup where a founder is doing it at midnight instead of building the product.

🔮 **Before you touch anything, predict:** given your company, do you want the copilot to be
**aggressive** (automate lots, accept some mistakes) or **cautious** (automate little, be almost always
right)? Write it down. You will check yourself in a minute.

## Step 3 · 🎛️ THE KNOB — set the confidence bar

In [ ]:
CONFIDENCE_BAR = START_BAR     # <-- CHANGE ME (any number from 0.30 to 1.00), then re-run


# ---------------------------------------------------------------- scoring
def weekly_cost(bar, show=True):
    auto = routed.confidence >= bar
    n_auto, n_human = auto.sum(), (~auto).sum()
    share_auto = n_auto / len(routed)
    acc = (routed.predicted[auto] == routed.category[auto]).mean() if n_auto else 0.0

    auto_week  = TICKETS_PER_WEEK * share_auto
    wrong_week = auto_week * (1 - acc)
    human_week = TICKETS_PER_WEEK - auto_week

    cost = wrong_week * COST_WRONG + human_week * COST_HUMAN
    if show:
        print(f"{NAME}  ·  confidence bar = {bar:.2f}\n")
        print(f"  routed automatically   {share_auto:6.0%}   ({auto_week:>6,.0f} tickets/week)")
        print(f"     ...of those, right   {acc:6.0%}")
        print(f"     ...of those, WRONG   {1-acc:6.0%}   ({wrong_week:>6,.0f} tickets/week)")
        print(f"  sent to a human        {1-share_auto:6.0%}   ({human_week:>6,.0f} tickets/week)")
        print()
        print(f"  cost of the mistakes   ${wrong_week*COST_WRONG:>10,.0f}")
        print(f"  cost of human sorting  ${human_week*COST_HUMAN:>10,.0f}")
        print(f"  {'-'*38}")
        print(f"  TOTAL PER WEEK         ${cost:>10,.0f}")
    return cost

weekly_cost(CONFIDENCE_BAR)

Now **change `CONFIDENCE_BAR` and run the cell again.**

Do not creep. Try **0.35** and then **0.95** first, so you can see the whole range, then close in on
the best value for your company.

Watch **which of the two cost lines moves** each time, and by how much. That is the whole game: for
your company, is it more expensive to be wrong, or more expensive to make a human do it?

## Step 4 · Submit your answer

In [ ]:
# Run this when your team has decided. Read the line out loud when called on.
cost = weekly_cost(CONFIDENCE_BAR, show=False)
print(f"TEAM {TEAM}  ·  {NAME:20}  ·  bar {CONFIDENCE_BAR:.2f}  ·  ${cost:,.0f} per week")

---
---

# 🔒 Stop here

Do not run anything below until your instructor says so.

---
---

## Step 5 · The reveal

In [ ]:
bars = np.round(np.arange(0.30, 1.001, 0.01), 2)
costs = [weekly_cost(b, show=False) for b in bars]
best_i = int(np.argmin(costs)); best_bar = bars[best_i]

plt.figure(figsize=(9, 4.6))
plt.plot(bars, costs, lw=3, color="#4F46E5")
plt.axvline(best_bar, ls="--", c="#059669", lw=2)
plt.axvline(CONFIDENCE_BAR, ls=":", c="#E11D48", lw=2)
plt.scatter([best_bar], [costs[best_i]], s=90, color="#059669", zorder=5)
plt.text(best_bar, costs[best_i]*1.06, f"  best: {best_bar:.2f}", color="#059669", fontweight="bold")
plt.text(CONFIDENCE_BAR, max(costs)*0.96, " you", color="#E11D48", fontweight="bold")
plt.xlabel("confidence bar"); plt.ylabel("cost per week ($)")
plt.title(f"{NAME}: what every setting would have cost you")
plt.grid(alpha=.25); plt.gca().set_axisbelow(True)
for s in ["top","right"]: plt.gca().spines[s].set_visible(False)
plt.tight_layout(); plt.show()

yours = weekly_cost(CONFIDENCE_BAR, show=False)
left_on_table = (yours - costs[best_i]) / costs[best_i]
print(f"  your bar   {CONFIDENCE_BAR:.2f}  ->  ${yours:,.0f} / week")
print(f"  best bar   {best_bar:.2f}  ->  ${costs[best_i]:,.0f} / week")
print(f"  worst bar  {bars[int(np.argmax(costs))]:.2f}  ->  ${max(costs):,.0f} / week")
print()
print(f"  YOUR SCORE: you left {left_on_table:.1%} on the table  (${yours-costs[best_i]:,.0f} a week)")
print(f"  Lowest percentage in the room wins.")

## Step 6 · Everyone's answer at once

In [ ]:
plt.figure(figsize=(10, 5.2))
palette = ["#0EA5E9","#059669","#D97706","#7C3AED","#E11D48","#334155"]
rows = []
for t,(nm,_,cw,ch,_st) in COMPANIES.items():
    cs=[]
    for b in bars:
        a = routed.confidence >= b
        acc = (routed.predicted[a]==routed.category[a]).mean() if a.sum() else 0.0
        aw = TICKETS_PER_WEEK*a.mean()
        cs.append(aw*(1-acc)*cw + (TICKETS_PER_WEEK-aw)*ch)
    i=int(np.argmin(cs))
    plt.plot(bars, np.array(cs)/1000, lw=2.4, color=palette[t-1], label=f"{t}. {nm}")
    plt.scatter([bars[i]],[cs[i]/1000], s=80, color=palette[t-1], zorder=5)
    a = routed.confidence >= bars[i]
    rows.append({"team":t,"company":nm,"$ wrong route":cw,"$ human sort":ch,
                 "BEST bar":bars[i],"automated":f"{a.mean():.0%}",
                 "cost/week":f"${cs[i]:,.0f}"})
plt.xlabel("confidence bar"); plt.ylabel("cost per week ($000s)")
plt.title("Same copilot. Same tickets. Six companies.")
plt.legend(fontsize=9.5, frameon=False); plt.grid(alpha=.25); plt.gca().set_axisbelow(True)
for s in ["top","right"]: plt.gca().spines[s].set_visible(False)
plt.tight_layout(); plt.show()

pd.DataFrame(rows).set_index("team")

## What just happened

Every team ran **the same copilot** on **the same 160 tickets** with **the same 64% accuracy**.
Nobody had a better model. Nobody had better data.

And the right answer ranged from **automate 96% of the queue** to **automate 12% of it**.

The only thing that changed was **what a mistake costs**.

### The three things to take out of the room

**1. There is no "best" setting for an AI system.** There is only the best setting for a specific
business. Anyone who quotes you a single accuracy number has not told you enough to make a decision.

**2. The question that unlocks it is "what does a wrong answer cost us here?"** Not "how accurate is
it?" You just watched that one question move the correct answer across the entire range. It is the
most useful question you can ask in a room where somebody is proposing an AI feature, and vendors
will almost never ask it for you.

**3. Sometimes the honest answer is "do not automate this."** Look at Halcyon Health. Even at its best
setting it only automates 12% of the queue, and the savings are thin. Push the cost of a mistake a
little higher and the correct decision becomes **do not ship it at all**. Knowing when to say that is
worth more than knowing how to build the thing.

### The one line to remember

> Accuracy is a property of the model. **The right threshold is a property of the business.**